# 7. Learn and evaluate a baseline

**Open:** restart R and save the supplied script as `scripts/my-07-baseline.R`.

**Input:** `outputs/checkpoints/farm-split.rds` from Chapter 6. **Result:** a score table, predictions and a plot you can explain.

## 7.1 Fit one transparent rule

Run the supplied `load-split` block to restore the training and test tables:

In [1]:
if (!file.exists("outputs/checkpoints/farm-split.rds")) stop("Run Chapter 6 first to create farm-split.rds.")
split_checkpoint <- readRDS("outputs/checkpoints/farm-split.rds")
train <- split_checkpoint$train
test <- split_checkpoint$test
farm_ids <- split_checkpoint$farm_ids
test_farms <- split_checkpoint$test_farms

Compute a mean using **training outcomes only**, then repeat it for each test record:

In [2]:
baseline_log10 <- mean(train$log10_msc)
baseline_log10
test$prediction_log10 <- rep(baseline_log10, nrow(test))
stopifnot(all(test$prediction_log10 == mean(train$log10_msc)))

[1] 0.7450425

The learned value is about **0.7450 log10 CFU/mL**. Every held-out sample receives it. No weather or management predictors enter this rule.

**Connect it to Chapter 5:** the invented example's mean log10 prediction was 1, equivalent to `10^1 = 10` CFU/mL. Here it is 0.7450, equivalent to `10^0.7450`, about **5.56 CFU/mL**.

Run the supplied `baseline-scale` block. It compares three calculations from the same training records:

In [3]:
# Interpret the fixed prediction using training records only; do not refit it.
data.frame(mean_log10 = baseline_log10,
           same_prediction_CFU_ml = 10^baseline_log10,
           arithmetic_mean_CFU_ml = mean(train$Conc))

mean_log10,same_prediction_CFU_ml,arithmetic_mean_CFU_ml
<dbl>,<dbl>,<dbl>
0.7450425,5.559587,16.7819


The original-scale arithmetic mean is about **16.78 CFU/mL**. The inverse log10 mean, 5.56, is the geometric mean of the stored training concentrations, including the below-limit substitutions. It expresses our fixed log10 prediction on another scale. If a future task asks for the average concentration on the original scale, this conversion alone does not answer it.

**Check:** can we replace the log10 prediction 0.7450 with 5.56 when scoring log10 observations? No: observations and predictions must use the same scale.

## 7.2 Read the result table

A residual is observed minus predicted. As in the hand example, MAE averages absolute residuals; RMSE averages their squares and takes a square root.

In [4]:
test$error_log10 <- test$log10_msc - test$prediction_log10
rmse <- sqrt(mean(test$error_log10^2))
mae <- mean(abs(test$error_log10))
data.frame(model = "Training-mean baseline", n_test = nrow(test),
           RMSE_log10_CFU_ml = rmse, MAE_log10_CFU_ml = mae)

model,n_test,RMSE_log10_CFU_ml,MAE_log10_CFU_ml
<chr>,<int>,<dbl>,<dbl>
Training-mean baseline,105,0.5289231,0.409148


**How to read these numbers:**

- **105:** the number of held-out test records, from 21 farms. Each record contributes one equally weighted error.
- **RMSE about 0.529; MAE about 0.409:** prediction errors on the **log10** target scale.
- **Their role:** a starting point for evaluating later procedures on an appropriate validation design.

To give a log10 residual some meaning, compare two invented predictions:

| Predicted CFU/mL | Observed CFU/mL | Original-scale difference | Observed / predicted | log10 residual |
|---:|---:|---:|---:|---:|
| 10 | 20 | 10 | 2 | about 0.301 |
| 100 | 200 | 100 | 2 | about 0.301 |

Both observations are twice their predictions, so their log10 residuals match even though their concentration differences do not. A log10 residual of 1 corresponds to an observed concentration ten times its prediction; −1 corresponds to one tenth.

RMSE 0.529 summarizes many residuals; it is **not a 52.9% error rate**. Nor does it say every sample has a concentration ratio of `10^0.529`. A single baseline score cannot tell us whether the error is acceptable for an application; that needs a suitable comparison and a defined error requirement.

```{admonition} Why a record and a farm can have different weights
:class: tip dropdown
Invented example: farm A has one record of 10, and farm B has three records of 30 CFU/mL. Averaging all records gives 25. Averaging the two farm means, 10 and 30, gives 20. These use different weights. Here each record contributes equally to the training mean and the test metrics; farms with more records therefore contribute more. Holding out whole farms does not itself give every farm equal weight.
```

## 7.3 Reconstruct one error yourself

Run the supplied `independent` check. It also displays three observed values, predictions and residuals:

In [5]:
rmse_check <- sqrt(sum((test$log10_msc - baseline_log10)^2) / nrow(test))
mae_check <- sum(abs(test$log10_msc - baseline_log10)) / nrow(test)
stopifnot(isTRUE(all.equal(rmse, rmse_check)))
stopifnot(isTRUE(all.equal(mae, mae_check)))
round(data.frame(observed = head(test$log10_msc, 3),
                 predicted = head(test$prediction_log10, 3),
                 residual = head(test$error_log10, 3)), 4)

observed,predicted,residual
<dbl>,<dbl>,<dbl>
0.8129,0.745,0.0679
0.5441,0.745,-0.2010
-0.3010,0.745,-1.0461


Choose the first row. Subtract predicted from observed by hand and compare with its residual. Display rounding can cause a small difference; the automated check uses full precision.

**Check the scope:** the three printed rows are a preview, not all 105 test records. Run `rmse-scope`:

In [6]:
# Three displayed records are a preview; the complete score uses all test records.
preview_errors <- head(test$error_log10, 3)
data.frame(records_used = c(length(preview_errors), nrow(test)),
           RMSE_log10 = c(sqrt(mean(preview_errors^2)), rmse))

records_used,RMSE_log10
<int>,<dbl>
3,0.6162426
105,0.5289231


The three-row RMSE is about **0.616**, while the full 105-row RMSE is about **0.529**. These are different calculations. The one-row subtraction above checks a residual; it cannot verify the full test score.

## 7.4 Save and inspect the evidence

Select **all remaining supplied blocks**, from `sensitivity` through `audit-save`, and Run. These are **required supplied blocks**. The target-limit calculation creates objects used by the export, so run it even if you leave the details folded. They then save the outputs. The blocks still run even if you leave their technical details collapsed.

**Required supplied calculation — `sensitivity`:** the same fixed predictions are evaluated on the quantified test subset. This describes another subset; it does not recover true below-detection concentrations or fix that limitation.

In [7]:
# Evaluation subset only: the baseline is not refitted or chosen using this result.
quantified_test <- test[!test$left_censored, ]
quantified_rmse <- sqrt(mean(quantified_test$error_log10^2))
data.frame(evaluation = c("All eligible test records", "Quantified test records only"),
           n = c(nrow(test), nrow(quantified_test)),
           RMSE_log10_CFU_ml = c(rmse, quantified_rmse))

evaluation,n,RMSE_log10_CFU_ml
<chr>,<int>,<dbl>
All eligible test records,105,0.5289231
Quantified test records only,104,0.5062914


**File export — `save`:**

In [8]:
dir.create("outputs/baseline", recursive = TRUE, showWarnings = FALSE)
score <- data.frame(model = "Training-mean baseline", training_mean_log10 = baseline_log10,
                    train_rows = nrow(train), test_rows = nrow(test),
                    train_farms = length(unique(train$FarmID)), test_farms = length(test_farms),
                    RMSE_log10_CFU_ml = rmse, MAE_log10_CFU_ml = mae,
                    quantified_test_n = nrow(quantified_test), quantified_test_RMSE = quantified_rmse)
write.csv(score, "outputs/baseline/scores.csv", row.names = FALSE)
split_table <- data.frame(FarmID = farm_ids,
                         split = ifelse(farm_ids %in% test_farms, "test", "train"))
write.csv(split_table, "outputs/baseline/farm-split.csv", row.names = FALSE)
write.csv(test[c("FarmID", "Sampling", "left_censored", "log10_msc", "prediction_log10", "error_log10")],
          "outputs/baseline/test-predictions.csv", row.names = FALSE)
writeLines(capture.output(sessionInfo()), "outputs/baseline/session-info.txt")
file.exists("outputs/baseline/scores.csv")

[1] TRUE

**Plot export — `plot`:**

In [9]:
png("outputs/baseline/training-distribution.png", width = 1200, height = 720, res = 150)
par(mar = c(4.5, 4.5, 2.5, 1), family = "sans", col.axis = "#294943", col.lab = "#294943")
hist(train$log10_msc, breaks = 18, col = "#8cc7b6", border = "white",
     main = "The baseline is learned from training farms",
     xlab = "Training log10 MSC (CFU/mL)", ylab = "Number of records")
abline(v = baseline_log10, col = "#a94c31", lwd = 3)
legend("topright", "Training mean", col = "#a94c31", lwd = 3, bty = "n")
invisible(dev.off())
cat("Saved outputs/baseline/training-distribution.png\n")

Saved outputs/baseline/training-distribution.png


![Training log10 MSC distribution with the training mean marked by a vertical line.](../outputs/baseline/training-distribution.png)

The plot is written directly to `outputs/baseline/training-distribution.png`. Open that file in Files; it may not appear in the Plots pane.

**Selection record — `audit-save`:**

In [10]:
write.csv(split_checkpoint$audit, "outputs/baseline/data-audit.csv", row.names = FALSE)
cat("All data and baseline checks passed.\n")

All data and baseline checks passed.


| Find in outputs/baseline | What you can check |
|---|---|
| `scores.csv` | The fitted mean and the two error metrics |
| `farm-split.csv` | Which role each farm was assigned |
| `test-predictions.csv` | Observed values, predictions and residuals |
| `training-distribution.png` | Where the learned mean lies in the training distribution |

## 7.5 Write a conclusion that fits the evidence

Write two sentences: name the rule and its test error, then state one important limit. Keep the target scale and held-out setting explicit.

```{admonition} Reference write-up — open after you try
:class: tip dropdown
A training-mean baseline achieved RMSE about 0.529 and MAE about 0.409 log10 CFU/mL on 105 records from 21 held-out farms. This is a fixed-split microbial-quality benchmark; below-detection target substitutions and the represented study setting limit its interpretation. It does not establish a causal effect or a milk-safety decision.
```

Different questions require different evidence:

| Question | What this exercise provides |
|---|---|
| How well does this rule predict MSC for held-out farms in the study? | A baseline error on the stated log10 target and fixed split. |
| Would changing a farm's management reduce MSC? | No intervention evidence; prediction alone does not establish a causal effect. |
| Is a particular bottle of milk safe? | No safety decision; an MSC prediction is not a complete safety assessment. |

**Explain:** why would even a lower predictive RMSE leave the last two questions unanswered?

## 7.6 Check the full score with AI

**Optional live activity:** keep the reference blocks intact. At the end of your my- script, the supplied work area contains comments only until you add code:

In [11]:
# My AI RMSE check: add your returned R code below in your my- copy.
# Use all rows of test, not just the three-row preview.

**Ask AI for code to run locally**, rather than asking it to infer the full score from three pasted rows:

```text
I am using base R. My object test already has 105 held-out records.
Observed log10 values are in test$log10_msc, and fixed predictions are in
test$prediction_log10. Write a small R block to print the row count and
calculate RMSE over ALL 105 rows. Do not refit a model, sample rows,
change the split or modify test. I will run the code locally.
```

In your my- script, find **My AI RMSE check** at the end and add the returned R code below that comment. After completing the earlier required blocks, run only your added lines. The object `test` already exists in this session; this check uses that full table directly. Check that the row count is 105 and RMSE is about 0.529. If you cannot use live chat, inspect this reference calculation and label your route as code inspection:

```r
nrow(test)
sqrt(mean((test$log10_msc - test$prediction_log10)^2))
```

The reference uses every row because neither column is shortened with `head()`. Keep your hand check and actual AI-use record separately from the supplied results.

**Finish:** locate your score, plot and AI-use record, explain one residual, then restart R and run a my- copy of `scripts/worked-example.R` from the beginning. It should recreate the real-data outputs.

**Next:** [could group averages make a useful tree?](08-first-tree.ipynb)